In [ ]:
import os
import sys
import time
import warnings
from datetime import datetime
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

# ============================================================================
# تنظیمات
# ============================================================================

# True  -> فقط افزایش نسبت به میانگین مهم است: Deviation = (X-μ)/σ
# False -> قدر مطلق: |Z| = |X-μ|/σ (افزایش و کاهش هر دو)
ONE_SIDED = True

# زمان‌های اجرای روزانه (فرمت HH:MM)
RUN_TIMES = ["11:10", "21:00"]

BASELINE_DAYS = 30
FAULT_DAYS = 30
DEVIATION_REF_DAYS = 5 * 365  # بازه مرجع برای Deviation_Score در RCA


# ============================================================================
# منطق وضعیت بر اساس Deviation
#   Dev < 2        Normal
#   2 <= Dev < 3   Watch
#   3 <= Dev < 4   Abnormal
#   4 <= Dev < 5   Severe
#   Dev >= 5       Critical statistical deviation
# ============================================================================

def deviation_to_status(dev):
    if pd.isna(dev):
        return 'No Data'
    if dev >= 6:
        return 'Critical statistical deviation'
    if dev >= 5:
        return 'Severe'
    if dev >= 4:
        return 'Abnormal'
    if dev >= 3:
        return 'Watch'
    return 'Normal'


def get_deviation_status(val, mean, std):
    if pd.isna(val) or pd.isna(std) or std == 0:
        return 'Normal'
    z = (val - mean) / std
    return deviation_to_status(z if ONE_SIDED else abs(z))


def get_sigma_status(val, mean, std):
    """منطق قبلی (بدون تغییر) - فقط برای ستون‌های Status_ در شیت Fault_Status.
    val > mean و فاصله > 5σ  → Action Required
    val > mean و فاصله > 3σ  → Warning
    val > mean و فاصله > 2σ  → Normal (Minor Change)
    غیر این صورت              → Normal
    """
    if std == 0 or pd.isna(std):
        return 'Normal'
    deviation = (val - mean) / std
    if deviation > 5:
        return 'Action Required'
    elif deviation > 3:
        return 'Warning'
    elif deviation > 2:
        return 'Normal (Minor Change)'
    else:
        return 'Normal'


def build_z_fault_detail(df_baseline, df_fault, sensors):
    """برای هر ردیف زمانی: مقدار واقعی سنسور، مقدار Z و وضعیت Z (شیت جدید)"""
    out = pd.DataFrame(index=df_fault.index)
    for col in sensors:
        if col not in df_baseline.columns or col not in df_fault.columns:
            continue
        mean, std = df_baseline[col].mean(), df_baseline[col].std()
        out[f'Value_{col}'] = df_fault[col]  # مقدار واقعی سنسور (کنار ستون Z)
        if pd.isna(std) or std == 0:
            out[f'Z_{col}'] = np.nan
            out[f'ZStatus_{col}'] = 'Normal'
            continue
        z = (df_fault[col] - mean) / std
        z_eff = z if ONE_SIDED else z.abs()
        out[f'Z_{col}'] = z_eff.round(3)
        out[f'ZStatus_{col}'] = z_eff.apply(deviation_to_status)
    return out.reset_index()


def build_sensor_status_summary(df_baseline, df_fault, sensors):
    """یک وضعیت برای هر ستون:
    Status           -> بر اساس آخرین مقدار ثبت‌شده
    Status_Worst_30d -> بدترین وضعیت در ۳۰ روز اخیر
    """
    rows = []
    for col in sensors:
        if col not in df_baseline.columns or col not in df_fault.columns:
            continue
        base = df_baseline[col].dropna()
        fault = df_fault[col].dropna()
        if base.empty or fault.empty:
            continue

        mean, std = base.mean(), base.std()
        if pd.isna(std) or std == 0:
            rows.append({'Sensor': col, 'Baseline_Mean': mean, 'Baseline_Std': std,
                         'Latest_Date': fault.index[-1], 'Latest_Value': fault.iloc[-1],
                         'Deviation_Latest': np.nan, 'Status': 'Normal',
                         'Deviation_Max_30d': np.nan, 'Status_Worst_30d': 'Normal'})
            continue

        z = (fault - mean) / std
        z_eff = z if ONE_SIDED else z.abs()
        rows.append({
            'Sensor': col,
            'Baseline_Mean': round(mean, 4),
            'Baseline_Std': round(std, 4),
            'Latest_Date': fault.index[-1],
            'Latest_Value': round(fault.iloc[-1], 4),
            'Deviation_Latest': round(z_eff.iloc[-1], 3),
            'Status': deviation_to_status(z_eff.iloc[-1]),
            'Deviation_Max_30d': round(z_eff.max(), 3),
            'Status_Worst_30d': deviation_to_status(z_eff.max()),
        })
    return pd.DataFrame(rows)


# ============================================================================
# تحلیل ۱: جامع (DBSCAN + Deviation + Partial Correlation + RCA)
# ============================================================================

def analysis_comprehensive(file_path, output_filename, label, all_features, target_sensors):
    print(f"\n{'='*60}\n🔄 [{label}-1] تحلیل جامع\n{'='*60}")

    try:
        df_raw = pd.read_excel(file_path)
        df_raw['date'] = pd.to_datetime(df_raw['date'])
        df_raw = df_raw.sort_values(by='date')
        print(f"✅ دیتا بارگذاری شد. تعداد: {len(df_raw):,}")
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False

    # --- پاکسازی با DBSCAN (حذف ۱۰٪ دورترین نقاط از مرکز خوشه) ---
    scaled_data = StandardScaler().fit_transform(df_raw[all_features])
    labels = DBSCAN(eps=0.5, min_samples=5).fit_predict(scaled_data)
    cluster_centers = {cid: scaled_data[labels == cid].mean(axis=0) for cid in set(labels) if cid != -1}

    def calculate_distance(i):
        label_i, point = labels[i], scaled_data[i].reshape(1, -1)
        if label_i != -1:
            return cdist(point, cluster_centers[label_i].reshape(1, -1))[0][0]
        return np.min(cdist(point, np.array(list(cluster_centers.values())))) if cluster_centers else 0.0

    df_raw['distance'] = [calculate_distance(i) for i in range(len(df_raw))]
    df_cleaned = df_raw.sort_values(by='distance', ascending=False).iloc[int(len(df_raw) * 0.1):].copy()
    df_cleaned = df_cleaned.sort_values(by='date').set_index('date')

    # --- بازه‌ها ---
    last_date = df_cleaned.index.max()
    start_analysis_date = last_date - pd.Timedelta(days=FAULT_DAYS)
    baseline_end = start_analysis_date - pd.Timedelta(days=1)
    baseline_start = baseline_end - pd.Timedelta(days=BASELINE_DAYS)

    df_baseline = df_cleaned[(df_cleaned.index >= baseline_start) & (df_cleaned.index <= baseline_end)]
    df_fault = df_cleaned[df_cleaned.index >= start_analysis_date].copy()

    # --- وضعیت هر ردیف بر اساس Deviation ---
    for col in target_sensors:
        if col in df_baseline.columns and col in df_fault.columns:
            m = df_baseline[col].mean()
            s = df_baseline[col].std()
            s = s if (pd.notna(s) and s != 0) else 1e-6
            df_fault[f'Status_{col}'] = df_fault[col].apply(lambda x: get_sigma_status(x, m, s))

    # --- محاسبات جدید Z (در شیت‌های جدید ذخیره می‌شوند) ---
    df_status_summary = build_sensor_status_summary(df_baseline, df_fault, target_sensors)
    df_z_detail = build_z_fault_detail(df_baseline, df_fault, target_sensors)

    # --- Partial Correlation ---
    def get_partial_corr(data, columns):
        corr_matrix = data[columns].corr().values
        precision = np.linalg.inv(corr_matrix + np.eye(corr_matrix.shape[0]) * 1e-6)
        d = np.sqrt(np.diag(precision))
        partial_corr = -precision / np.outer(d, d)
        np.fill_diagonal(partial_corr, 1.0)
        return pd.DataFrame(partial_corr, index=columns, columns=columns)

    pcorr_baseline = get_partial_corr(df_baseline, all_features)
    pcorr_fault = get_partial_corr(df_fault[all_features], all_features)
    delta_pcorr = pcorr_fault - pcorr_baseline

    # --- Deviation Score برای RCA (بازه مرجع بلندتر) ---
    dev_ref_end = start_analysis_date - pd.Timedelta(days=1)
    dev_ref_start = dev_ref_end - pd.Timedelta(days=DEVIATION_REF_DAYS)
    df_dev_ref = df_cleaned[(df_cleaned.index >= dev_ref_start) & (df_cleaned.index <= dev_ref_end)]

    deviation_scores = {}
    for c in all_features:
        reference_data = df_dev_ref[c].dropna() if c in df_dev_ref.columns else pd.Series(dtype=float)
        current_data = df_fault[c].dropna() if c in df_fault.columns else pd.Series(dtype=float)
        if reference_data.empty or current_data.empty:
            deviation_scores[c] = 0.0
            continue
        ref_std = reference_data.std()
        deviation_scores[c] = (abs(current_data.mean() - reference_data.mean()) / ref_std
                               if pd.notna(ref_std) and ref_std > 0 else 0.0)

    change_scores = {c: np.mean([abs(delta_pcorr.loc[c, o]) for o in all_features if o != c])
                     for c in all_features}

    def normalize_dict(d):
        vals = np.array(list(d.values()))
        if vals.max() == vals.min():
            return {k: 0.5 for k in d.keys()}
        return {k: (v - vals.min()) / (vals.max() - vals.min()) for k, v in d.items()}

    dev_norm = normalize_dict(deviation_scores)
    chg_norm = normalize_dict(change_scores)

    rca_list = []
    for c in all_features:
        strength_fault = np.mean([abs(pcorr_fault.loc[c, o]) for o in all_features if o != c])
        score = dev_norm[c] * 0.5 + chg_norm[c] * 0.5
        rca_list.append({
            'Sensor': c, 'Change_Normalized': chg_norm[c], 'Change_Score_Delta': change_scores[c],
            'Deviation_Normalized': dev_norm[c], 'Deviation_Score': deviation_scores[c],
            'Strength_Score_Fault': strength_fault, 'RCA_Score': score,
            'Root_Cause_Suspicion': score * (1 - strength_fault)
        })
    df_rca_summary = pd.DataFrame(rca_list).sort_values(by='Root_Cause_Suspicion', ascending=False)

    # --- ذخیره ---
    try:
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
            status_cols = [f'Status_{c}' for c in target_sensors if f'Status_{c}' in df_fault.columns]
            cols_to_save = [c for c in target_sensors if c in df_fault.columns] + status_cols
            if cols_to_save:
                df_fault[cols_to_save].reset_index().to_excel(writer, sheet_name='Fault_Status', index=False)
            df_rca_summary.to_excel(writer, sheet_name='RCA_Summary', index=False)
            delta_pcorr.to_excel(writer, sheet_name='Delta_Correlation_Matrix')
            # --- شیت‌های جدید (بعد از شیت‌های قبلی) ---
            df_status_summary.to_excel(writer, sheet_name='Z_Sensor_Status', index=False)
            df_z_detail.to_excel(writer, sheet_name='Z_Fault_Detail', index=False)
        print(f"✅ ذخیره شد: {output_filename}")
        return True
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False


# ============================================================================
# تحلیل ۲: EWMA + حدود کنترل + RCA
# ============================================================================

def analysis_ewma(file_path, output_filename, label, target_sensors):
    print(f"\n{'='*60}\n🔄 [{label}-2] تحلیل EWMA\n{'='*60}")
    LAMBDA = 0.2

    try:
        df = pd.read_excel(file_path, parse_dates=['date'])
        df = df.sort_values('date').set_index('date')
        print(f"✅ دیتا بارگذاری شد. تعداد: {len(df):,}")
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False

    last_date = df.index.max()
    split_date = last_date - pd.Timedelta(days=FAULT_DAYS)
    baseline_start = split_date - pd.Timedelta(days=BASELINE_DAYS)

    df_baseline = df.loc[baseline_start:split_date].copy()
    df_fault = df.loc[split_date:last_date].copy()

    def calculate_ewma(data, lambda_val):
        ewma_values = np.zeros(len(data))
        if len(data) == 0:
            return ewma_values
        ewma_values[0] = data[0]
        for t in range(1, len(data)):
            ewma_values[t] = lambda_val * data[t] + (1 - lambda_val) * ewma_values[t - 1]
        return ewma_values

    def calculate_control_limits(mean, std, lambda_val):
        factor = 3 * std * np.sqrt(lambda_val / (2 - lambda_val))
        return mean + factor, mean - factor

    results = []
    for col in target_sensors:
        if col not in df_fault.columns or col not in df_baseline.columns:
            continue
        mean_base = df_baseline[col].mean()
        std_base = df_baseline[col].std()
        ucl, lcl = calculate_control_limits(mean_base, std_base, LAMBDA)
        ewma_values = calculate_ewma(df_fault[col].values, LAMBDA)
        current_ewma = ewma_values[-1]

        time_diff_series = df_fault.index.to_series().diff().dt.total_seconds() / 3600.0
        instant_slopes = df_fault[col].diff() / time_diff_series
        avg_slope = instant_slopes.mean()
        current_raw_value = df_fault[col].iloc[-1]

        if avg_slope <= 0 or current_raw_value >= ucl:
            hours_to_ucl = 0.0
        else:
            hours_to_ucl = round((ucl - current_raw_value) / avg_slope, 2)
        if np.isnan(hours_to_ucl) or np.isinf(hours_to_ucl):
            hours_to_ucl = 0.0

        out_of_control = "Yes ⚠️" if (current_ewma > ucl or current_ewma < lcl) else "No"
        results.append({
            'Sensor': col, 'Current_Raw_Value': round(current_raw_value, 4),
            'Current_EWMA': round(current_ewma, 4), 'UCL': round(ucl, 4),
            'LCL': round(lcl, 4), 'Baseline_Mean': round(mean_base, 4),
            'Baseline_Std': round(std_base, 4),
            'Average_Slope_per_Hour': round(avg_slope, 6),
            'Hours_to_UCL': hours_to_ucl, 'Out_Of_Control': out_of_control
        })

    numeric_cols = df_baseline.select_dtypes(include=[np.number]).columns.tolist()
    rca_list = []
    for c in numeric_cols:
        if c not in df_fault.columns:
            continue
        mean_base = df_baseline[c].mean()
        std_base = df_baseline[c].std()
        mean_fault = df_fault[c].mean()
        dev = abs(mean_fault - mean_base) / std_base if std_base > 0 else 0
        rca_list.append({'Sensor': c, 'Baseline_Mean': round(mean_base, 4),
                         'Fault_Mean': round(mean_fault, 4), 'Deviation_Score': round(dev, 4)})
    rca_summary = pd.DataFrame(rca_list).sort_values('Deviation_Score', ascending=False)

    try:
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
            pd.DataFrame(results).to_excel(writer, sheet_name='Speed_Analysis', index=False)
            rca_summary.to_excel(writer, sheet_name='RCA_Ranking', index=False)
            pd.DataFrame({
                'Parameter': ['Lambda (λ)', 'UCL Formula', 'LCL Formula', 'EWMA Formula', 'Hours_to_UCL Rule'],
                'Value': [f'{LAMBDA}', 'μ + 3σ√(λ/(2-λ))', 'μ - 3σ√(λ/(2-λ))',
                          'E_t = λ·X_t + (1-λ)·E_{t-1}',
                          'If slope>0 and value<UCL → (UCL-value)/slope | Else → 0']
            }).to_excel(writer, sheet_name='Methodology', index=False)
        print(f"✅ ذخیره شد: {output_filename}")
        return True
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False


# ============================================================================
# تحلیل ۳: EWMA با پاکسازی DBSCAN
#   extended=False -> فقط حد بالا (رفتار قبلی توربین)
#   extended=True  -> حد بالا و پایین و تعداد داده‌ها (رفتار قبلی ژنراتور و روغن‌کاری)
# ============================================================================

def analysis_ewma_dbscan(file_path, output_filename, label, target_sensors, extended):
    print(f"\n{'='*60}\n🔄 [{label}-3] EWMA با DBSCAN\n{'='*60}")

    try:
        df = pd.read_excel(file_path, parse_dates=['date'])
        df = df.sort_values('date')
        print(f"✅ دیتا بارگذاری شد. تعداد: {len(df):,}")
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False

    last_date = df['date'].max()
    fault_start = last_date - pd.Timedelta(days=FAULT_DAYS)
    baseline_start = fault_start - pd.Timedelta(days=BASELINE_DAYS)

    df_baseline = df[(df['date'] >= baseline_start) & (df['date'] < fault_start)].copy()
    df_fault = df[df['date'] >= fault_start].copy()

    def remove_outliers_dbscan(series):
        if series.empty or len(series) < 5:
            return series
        X = series.values.reshape(-1, 1)
        std_val = series.std()
        eps = 0.1 if (pd.isna(std_val) or std_val == 0) else max(std_val * 0.5, 0.01)
        min_samples = max(min(5, len(series) - 1), 1)
        clusters = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(X)
        return series if (clusters == -1).all() else series[clusters != -1]

    sensor_stats = {}
    if extended:
        for col in target_sensors:
            if col not in df.columns or col not in df_baseline.columns:
                continue
            clean_baseline = remove_outliers_dbscan(df_baseline[col].dropna())
            if not clean_baseline.empty:
                mean_val, std_val = clean_baseline.mean(), clean_baseline.std()
                sensor_stats[col] = {'mean': round(mean_val, 4), 'std': round(std_val, 4),
                                     'upper_band': round(mean_val + 5 * std_val, 4),
                                     'lower_band': round(mean_val - 5 * std_val, 4)}

    ewma_list, baseline_stats = [], []
    for col in target_sensors:
        if col not in df.columns:
            continue
        temp_fault = df_fault[['date', col]].copy()
        if not temp_fault.empty:
            temp_fault = temp_fault.rename(columns={col: 'Value'})
            temp_fault['AssetID'] = col
            temp_fault['EWMA'] = temp_fault['Value'].ewm(alpha=0.2, adjust=False).mean()
            if extended:
                if col in sensor_stats:
                    temp_fault['Repeated_Mean'] = sensor_stats[col]['mean']
                    temp_fault['Repeated_Std'] = sensor_stats[col]['std']
                    temp_fault['upper_band'] = sensor_stats[col]['upper_band']
                    temp_fault['lower_band'] = sensor_stats[col]['lower_band']
                else:
                    for f in ['Repeated_Mean', 'Repeated_Std', 'upper_band', 'lower_band']:
                        temp_fault[f] = np.nan
                ewma_list.append(temp_fault[['date', 'AssetID', 'Value', 'EWMA',
                                             'Repeated_Mean', 'Repeated_Std', 'upper_band', 'lower_band']])
            else:
                ewma_list.append(temp_fault[['date', 'AssetID', 'Value', 'EWMA']])

        if col in df_baseline.columns:
            clean_baseline = remove_outliers_dbscan(df_baseline[col].dropna())
            if not clean_baseline.empty:
                mean_val, std_val = clean_baseline.mean(), clean_baseline.std()
                row = {'AssetID': col, 'Clean_Mean': round(mean_val, 4),
                       'Clean_Std': round(std_val, 4),
                       'Normal_Upper_Limit(3Sigma)': round(mean_val + 5 * std_val, 4)}
                if extended:
                    row['Normal_Lower_Limit(3Sigma)'] = round(mean_val - 5 * std_val, 4)
                    row['Outliers_Removed'] = len(df_baseline[col]) - len(clean_baseline)
                    row['Baseline_Data_Count'] = len(clean_baseline)
                    row['Original_Data_Count'] = len(df_baseline[col].dropna())
                else:
                    row['Outliers_Removed'] = len(df_baseline[col]) - len(clean_baseline)
                baseline_stats.append(row)

    try:
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
            if ewma_list:
                pd.concat(ewma_list, ignore_index=True).to_excel(writer, sheet_name='EWMA_Comparison', index=False)
            if baseline_stats:
                pd.DataFrame(baseline_stats).to_excel(writer, sheet_name='Baseline_Stats', index=False)
        print(f"✅ ذخیره شد: {output_filename}")
        return True
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False


# ============================================================================
# تحلیل ۴: Dual EWMA (سریع/کند)
# ============================================================================

def analysis_dual_ewma(file_path, output_filename, label, target_sensors):
    print(f"\n{'='*60}\n🔄 [{label}-4] Dual EWMA\n{'='*60}")

    try:
        df = pd.read_excel(file_path, parse_dates=['date'])
        df = df.sort_values('date')
        print(f"✅ دیتا بارگذاری شد. تعداد: {len(df):,}")
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False

    last_date = df['date'].max()
    df_recent = df[df['date'] >= last_date - pd.Timedelta(days=30)].copy()

    alpha_fast, alpha_slow = 0.22, 0.035
    results_list = []
    for col in target_sensors:
        if col in df_recent.columns:
            t = df_recent[['date', col]].copy().rename(columns={col: 'Raw_Value'})
            t['AssetID'] = col
            t['Daily_EWMA_Fast'] = t['Raw_Value'].ewm(alpha=alpha_fast, adjust=False).mean()
            t['Weekly_EWMA_Slow'] = t['Raw_Value'].ewm(alpha=alpha_slow, adjust=False).mean()
            t['Signal_Gap'] = t['Daily_EWMA_Fast'] - t['Weekly_EWMA_Slow']
            t['Deviation_Percent'] = (t['Signal_Gap'] / t['Weekly_EWMA_Slow']) * 100
            results_list.append(t[['date', 'AssetID', 'Raw_Value', 'Daily_EWMA_Fast',
                                   'Weekly_EWMA_Slow', 'Signal_Gap', 'Deviation_Percent']])

    try:
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
            if results_list:
                pd.concat(results_list, ignore_index=True).to_excel(
                    writer, sheet_name='Maintenance_Strategy', index=False)
        print(f"✅ ذخیره شد: {output_filename}")
        return True
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False


# ============================================================================
# تعریف مجموعه‌ها (سنسورها و مسیرها) و وظایف
# ============================================================================

OUT_ROOT = r'D:\data driven predictive maintenance\Techniques\Univariate Statistical Monitoring\outputs\G11'

SYSTEMS = [
    {
        'label': 'Turbine',
        'input': r'second_stage_inputs\G11\dsas_g11_turbine_bearings_output.xlsx',
        'output_base': OUT_ROOT + r'\dsas_g11_bearings_vibration_temp_univariate\univariate',
        'all_features': ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361',
                         'AssetID_9368', 'AssetID_9369', 'AssetID_9370',
                         'AssetID_9343', 'AssetID_9344', 'AssetID_9408'],
        'comp_targets': ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361'],
        'ewma_targets': ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361'],
        'extended_dbscan': False,
    },
    {
        'label': 'Generator',
        'input': r'second_stage_inputs\G11\dsas_g11_generator_bearings_output.xlsx',
        'output_base': OUT_ROOT + r'\dsas_g11_generator_bearings_univariate\univariate',
        'all_features': ['AssetID_9343', 'AssetID_9344', 'AssetID_9357', 'AssetID_9362', 'AssetID_9363',
                         'AssetID_9364', 'AssetID_9365', 'AssetID_9366', 'AssetID_9367', 'AssetID_9371',
                         'AssetID_9372', 'AssetID_9373'],
        'comp_targets': ['AssetID_9362', 'AssetID_9363', 'AssetID_9364', 'AssetID_9365',
                         'AssetID_9366', 'AssetID_9367', 'AssetID_9371', 'AssetID_9372', 'AssetID_9373'],
        'ewma_targets': ['AssetID_9362', 'AssetID_9363', 'AssetID_9364', 'AssetID_9365',
                         'AssetID_9366', 'AssetID_9367', 'AssetID_9371', 'AssetID_9372', 'AssetID_9373'],
        'extended_dbscan': True,
    },
    {
        'label': 'Lubrication',
        'input': r'second_stage_inputs\G11\dsas_g11_lubrication_system_output.xlsx',
        'output_base': OUT_ROOT + r'\dsas_g11_lubrication_system_univariate\univariate',
        'all_features': ['AssetID_9343', 'AssetID_9375', 'AssetID_8341', 'AssetID_8342', 'AssetID_8343',
                         'AssetID_8344', 'AssetID_8346', 'AssetID_9286', 'AssetID_9287'],
        'comp_targets': ['AssetID_9343', 'AssetID_9357', 'AssetID_9375', 'AssetID_8341', 'AssetID_8343',
                         'AssetID_8344', 'AssetID_8346', 'AssetID_9286', 'AssetID_9287'],
        'ewma_targets': ['AssetID_9375', 'AssetID_8341', 'AssetID_8343', 'AssetID_8344',
                         'AssetID_8346', 'AssetID_9286', 'AssetID_9287'],
        'extended_dbscan': True,
    },
]


def get_all_jobs():
    jobs = []
    for s in SYSTEMS:
        lab, inp, base = s['label'], s['input'], s['output_base']
        jobs += [
            {'name': f'{lab}-1: Comprehensive', 'file_path': inp, 'output_filename': f'{base}1.xlsx',
             'function': lambda f, o, s=s: analysis_comprehensive(
                 f, o, s['label'], s['all_features'], s['comp_targets'])},
            {'name': f'{lab}-2: EWMA', 'file_path': inp, 'output_filename': f'{base}2.xlsx',
             'function': lambda f, o, s=s: analysis_ewma(f, o, s['label'], s['ewma_targets'])},
            {'name': f'{lab}-3: EWMA+DBSCAN', 'file_path': inp, 'output_filename': f'{base}3.xlsx',
             'function': lambda f, o, s=s: analysis_ewma_dbscan(
                 f, o, s['label'], s['ewma_targets'], s['extended_dbscan'])},
            {'name': f'{lab}-4: Dual EWMA', 'file_path': inp, 'output_filename': f'{base}4.xlsx',
             'function': lambda f, o, s=s: analysis_dual_ewma(f, o, s['label'], s['ewma_targets'])},
        ]
    return jobs


def run_all_analyses():
    print("\n" + "=" * 80)
    print("🚀 شروع اجرای همه تحلیل‌ها (۳ مجموعه × ۴ تحلیل = ۱۲ وظیفه)")
    print(f"📅 زمان: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)

    jobs = get_all_jobs()
    results = []
    for i, job in enumerate(jobs, 1):
        print(f"\n{'#'*80}\n# وظیفه {i} از {len(jobs)}: {job['name']}")
        print(f"# ورودی: {job['file_path']}\n# خروجی: {job['output_filename']}\n{'#'*80}")
        try:
            success = job['function'](job['file_path'], job['output_filename'])
            results.append({'job_name': job['name'], 'success': success,
                            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')})
            print(f"{'✅' if success else '❌'} وظیفه {i} {'با موفقیت کامل شد' if success else 'با شکست مواجه شد'}")
        except Exception as e:
            print(f"❌ خطای غیرمنتظره در وظیفه {i}: {e}")
            results.append({'job_name': job['name'], 'success': False,
                            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'), 'error': str(e)})

    print("\n" + "=" * 80 + "\n📊 گزارش نهایی")
    ok = sum(1 for r in results if r['success'])
    print(f"✅ موفق: {ok} از {len(results)}\n❌ ناموفق: {len(results) - ok} از {len(results)}")
    for i, r in enumerate(results, 1):
        print(f"   {i:2d}. {'✅' if r['success'] else '❌'} {r['job_name']} - {r['timestamp']}")
    print("=" * 80)
    return results


# ============================================================================
# زمان‌بندی
# ============================================================================

def run_scheduler():
    print("=" * 80)
    print("🔄 برنامه زمان‌بندی خودکار شروع به کار کرد")
    print(f"⏰ زمان‌های اجرا (هر روز): {', '.join(RUN_TIMES)}")
    print("💡 برای توقف: Ctrl+C")
    print("=" * 80)

    last_run_times = {}
    while True:
        try:
            now = datetime.now()
            current_time = now.strftime("%H:%M")
            if current_time in RUN_TIMES and last_run_times.get(current_time) != now.strftime("%Y-%m-%d"):
                print(f"\n⏰ زمان اجرا فرا رسید: {now.strftime('%Y-%m-%d %H:%M:%S')}")
                run_all_analyses()
                last_run_times[current_time] = now.strftime("%Y-%m-%d")
                time.sleep(60)
            time.sleep(30)
        except KeyboardInterrupt:
            print(f"\n⏹️ برنامه متوقف شد: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
            break
        except Exception as e:
            print(f"❌ خطا در حلقه اصلی: {e}")
            time.sleep(60)


if __name__ == "__main__":
    try:
        # python predictive_maintenance_full.py --now   -> اجرای فوری یک‌باره
        # python predictive_maintenance_full.py         -> اجرای زمان‌بندی‌شده
        if '--now' in sys.argv:
            run_all_analyses()
        else:
            run_scheduler()
    except Exception as e:
        print(f"❌ خطای غیرمنتظره: {e}")
        import traceback
        traceback.print_exc()
        input("برای خروج Enter بزنید...")

🔄 برنامه زمان‌بندی خودکار شروع به کار کرد
⏰ زمان‌های اجرا (هر روز): 11:10, 21:00
💡 برای توقف: Ctrl+C

⏰ زمان اجرا فرا رسید: 2026-09-24 11:10:21

🚀 شروع اجرای همه تحلیل‌ها (۳ مجموعه × ۴ تحلیل = ۱۲ وظیفه)
📅 زمان: 2026-09-24 11:10:21

################################################################################
# وظیفه 1 از 12: Turbine-1: Comprehensive
# ورودی: second_stage_inputs\G11\dsas_g11_turbine_bearings_output.xlsx
# خروجی: D:\data driven predictive maintenance\Techniques\Univariate Statistical Monitoring\outputs\G11\dsas_g11_bearings_vibration_temp_univariate\univariate1.xlsx
################################################################################

🔄 [Turbine-1] تحلیل جامع
✅ دیتا بارگذاری شد. تعداد: 12,585
✅ ذخیره شد: D:\data driven predictive maintenance\Techniques\Univariate Statistical Monitoring\outputs\G11\dsas_g11_bearings_vibration_temp_univariate\univariate1.xlsx
✅ وظیفه 1 با موفقیت کامل شد

#######################################################################